# BigBird with a classification head

Fine-tunes `google/bigbird-roberta-base` to decide whether two texts were written by the same
author. The hyperparameters are tuned with Optuna on a sample of the training pairs. The final
model is trained on the training pairs, its threshold is selected on the validation pairs and
it is evaluated once on the test pairs.

A restarted run continues where it stopped: the study lives in a SQLite file, every training
writes checkpoints and a finished run is not repeated. A study is never continued with other
settings than the ones it was started with.

In [ ]:
SMOKE = False  # True - a few rows of every data set, a checkpoint after every step
N_TRIALS = 30  # finished trials of the study (complete; nothing is pruned)
SEED = 42  # seed of the tuning sample, the sampler, the new head and the training order
RUN_TAG = (
    ""  # label in the file names of a repeated run, so two runs never share a file
)
MODEL_ID = (
    "google/bigbird-roberta-base"  # Hugging Face id, or the folder of a saved model
)
DEVICE = "cuda"  # "cpu" only for the CPU checks - there is no fallback to the CPU

Standard libraries

In [ ]:
import gc
import logging
import re
import time
from functools import partial
from pathlib import Path

Data, plots and Optuna

In [ ]:
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
from optuna.trial import TrialState
from sklearn.metrics import confusion_matrix

Torch and Hugging Face

In [ ]:
import torch
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    set_seed,
)

The shared modules of the thesis

In [ ]:
from avcore import (
    data,
    evaluation,
    progress,
    runs,
    tracking,
    transformer_training,
    tuning,
)

# Configuration

The model - its name, the extra load options and the classification type

In [ ]:
MODEL_NAME = "bigbird-roberta-base"
# the run and its files are named after MODEL_NAME - the id of another model stops the run
if Path(MODEL_ID).name != MODEL_NAME:
    raise ValueError(
        f"MODEL_ID {MODEL_ID!r} does not end with MODEL_NAME {MODEL_NAME!r}."
    )
# extra from_pretrained options of the model, also part of the study settings
MODEL_LOAD_KWARGS = {}
CLASSIFICATION_TYPE = "classification_head"
DEVELOPER_INITIALS = "JP"

What the run is, in the words of the result tables

In [ ]:
RUN_IDENTITY = {
    "family": "transformer",
    "model": MODEL_NAME,
    "representation": "none",
    "combination": "none",
    "classifier": CLASSIFICATION_TYPE,
}

Search space - the tuned hyperparameters, the fixed warmup, the pruner and the threshold grid

In [ ]:
LEARNING_RATE_RANGE = (1e-6, 5e-5)
EPOCHS_RANGE = (1, 3)
BATCH_SIZE_CHOICES = [2, 4, 8]
WEIGHT_DECAY_RANGE = (0.0, 0.1)
# not searched - the warmup of the classification fine-tuning of the model papers
WARMUP_RATIO = 0.1
# no pruning - epochs is searched, so a pruner could only decide after epoch 2
PRUNER = optuna.pruners.NopPruner()
CLASSIFICATION_THRESHOLDS = [x / 1000 for x in range(100, 999)]

Sizes - the tuning sample, the checkpoint interval and the smoke run

In [ ]:
TRAIN_TUNING_SIZE = 19_999
CHECKPOINT_MINUTES = transformer_training.CHECKPOINT_MINUTES
FINAL_CHECKPOINT_KEY = "final"
SMOKE_ROWS = {"train": 32, "validation": 8, "test": 12}
if SMOKE:
    TRAIN_TUNING_SIZE = 16
    CHECKPOINT_MINUTES = 0

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
TRAIN_PATH = DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE
VALIDATION_PATH = DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE
TEST_PATH = DATA_DIR / "processed" / data.TEST_CLEANED_FILE
DATA_PATHS = [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]

The run name - it is part of every file name of the run

In [ ]:
run_name_parts = [DEVELOPER_INITIALS, MODEL_NAME, CLASSIFICATION_TYPE]
if SMOKE:
    run_name_parts.append("smoke")
if RUN_TAG:
    run_name_parts.append(str(RUN_TAG))
RUN_NAME = "_".join([*run_name_parts, "v2"])
if not re.fullmatch(r"[\w.-]+", RUN_NAME):
    raise ValueError(
        f"Run name {RUN_NAME!r} is not a plain file name - change RUN_TAG."
    )

Every file the run writes


In [ ]:
STUDY_PATH = DATA_DIR / "studies" / f"optuna_study_{RUN_NAME}.db"
MODEL_DIR = DATA_DIR / "models" / f"{RUN_NAME}_best_model"
# checkpoints and results are written outside the repository; the result tables are built from
# results_v2, so a smoke run writes into a folder of its own, one per RUN_TAG
CHECKPOINT_DIR = Path.home() / "checkpoints_v2" / RUN_NAME
RESULTS_DIR = Path.home() / "results_v2"
if SMOKE:
    RESULTS_DIR = Path.home() / "results_v2_smoke" / str(RUN_TAG)
RUN_DIR = RESULTS_DIR / RUN_NAME
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

Run context - the commit, the hashes of the lock file and of the data, the versions and the GPU.
It refuses a repository with uncommitted or unpushed changes.

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
# the training reports every epoch and every resume at this level
logging.getLogger("avcore").setLevel(logging.INFO)

device = torch.device(DEVICE)
# an unusable device stops the run here, before anything is read or written
torch.empty(0, device=device)
run_context = tracking.collect_run_context(PROJECT_DIR, DATA_PATHS)
print(f"Run {RUN_NAME} on {device}, commit {run_context.environment['commit'][:7]}")
# where the run stands, for watch_progress.py on dt - the run never reads it back
progress_file = progress.ProgressFile(
    Path.home() / "logs" / f"{RUN_NAME}.progress.json", RUN_NAME
)
progress_file.start_stage("loading")

# Load data

In [ ]:
train_df = data.read_jsonl(TRAIN_PATH)
validation_df = data.read_jsonl(VALIDATION_PATH)
test_df = data.read_jsonl(TEST_PATH)
# a prediction table is keyed by the pair id - checked on the whole files, before any training
for name, pairs_df in {"validation": validation_df, "test": test_df}.items():
    if not pairs_df["id"].is_unique:
        raise ValueError(f"The {name} pairs have repeated ids.")
if SMOKE:
    train_df = train_df.head(SMOKE_ROWS["train"])
    validation_df = validation_df.head(SMOKE_ROWS["validation"])
    test_df = test_df.head(SMOKE_ROWS["test"])

#### Create a subset of training data for finetuning

In [ ]:
tuning_df = train_df.sample(n=TRAIN_TUNING_SIZE, random_state=SEED).reset_index(
    drop=True
)
print(
    f"train {len(train_df)}, tuning {len(tuning_df)}, validation {len(validation_df)}, test {len(test_df)}"
)

# Set up model

The tokenizer and the config of the pretrained model are loaded once.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
pretrained_config = AutoConfig.from_pretrained(MODEL_ID)

The model is loaded again for every trial and for the final training, with the seed set first, so its new classification head always starts from the same weights. Every dropout stays as in the config of the pretrained model.

In [ ]:
def build_model(device: torch.device) -> PreTrainedModel:
    """Load the pretrained model with a new 2-label classification head.

    Args:
        device: Device the model is moved to.

    Returns:
        The model on the device, 0 = different author, 1 = same author.
    """
    # the new head gets the same weights on every call
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        problem_type="single_label_classification",
        **MODEL_LOAD_KWARGS,
    )
    return model.to(device)

# Hyperparameter optimization

Objective function - trains one trial on the tuning sample, reports the weighted F1 after
every epoch (no pruning) and returns the unrounded F1 at the best classification threshold on the
validation pairs.

In [ ]:
def objective(
    trial: optuna.Trial,
    *,
    tokenizer: PreTrainedTokenizerBase,
    tuning_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    device: torch.device,
    progress_file: progress.ProgressFile,
) -> float:
    """Train and evaluate one trial.

    Args:
        trial: Trial started by tuning.run_study.
        tokenizer: Tokenizer of the model.
        tuning_df: Training pairs of the tuning.
        validation_df: Pairs for the per-epoch report and the threshold search.
        device: Device of the training.
        progress_file: Shows the trial, the finished trials and the steps of the training.

    Returns:
        The unrounded F1 (a fraction) at the best classification threshold on the validation
        pairs; the rounded percent metrics are stored as user attributes.
    """
    progress_file.start_stage("tuning", **progress.trial_fields(trial, N_TRIALS))
    learning_rate = trial.suggest_float("lr", *LEARNING_RATE_RANGE, log=True)
    epochs = trial.suggest_int("epochs", *EPOCHS_RANGE)
    batch_size = trial.suggest_categorical("batch_size", BATCH_SIZE_CHOICES)
    weight_decay = trial.suggest_float("weight_decay", *WEIGHT_DECAY_RANGE)
    settings = transformer_training.TrainingSettings(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        epochs=epochs,
        batch_size=batch_size,
        warmup_ratio=WARMUP_RATIO,
        seed=SEED,
    )

    # a trial that ended with an exception leaves its model in a reference cycle of it
    gc.collect()
    torch.cuda.empty_cache()
    model = build_model(device)
    result = transformer_training.train_classifier(
        model=model,
        tokenizer=tokenizer,
        train_df=tuning_df,
        settings=settings,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_key=transformer_training.checkpoint_key_for(trial),
        validation_df=validation_df,
        trial=trial,
        checkpoint_minutes=CHECKPOINT_MINUTES,
        on_step=progress_file.update,
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()

    validation_scores_df = result.validation_scores_df
    threshold_results_df = evaluation.evaluate_classification_thresholds(
        validation_scores_df["same"].astype(int),
        validation_scores_df["score"],
        CLASSIFICATION_THRESHOLDS,
    )
    best_row = evaluation.select_best_threshold(threshold_results_df)
    trial.set_user_attr("threshold", float(best_row["threshold"]))
    for metric in evaluation.METRIC_COLUMNS:
        trial.set_user_attr(f"validation_{metric}", float(best_row[metric]))
    print(
        f"Trial {trial.number}: validation F1 {best_row['f1']} at threshold {best_row['threshold']}"
    )
    return float(best_row["f1_unrounded"])

The settings of the run - everything that is fixed before the first trial

In [ ]:
run_settings = {
    **RUN_IDENTITY,
    "model_id": MODEL_ID,
    "model_revision": pretrained_config._commit_hash,
    "model_load_kwargs": repr(MODEL_LOAD_KWARGS),
    "smoke": SMOKE,
    "seed": SEED,
    "n_trials": N_TRIALS,
    "mlflow_experiment": EXPERIMENT_NAME,
    "study_name": STUDY_PATH.stem,
    "max_length": transformer_training.MAX_LENGTH,
    "train_pairs": len(train_df),
    "tuning_pairs": len(tuning_df),
    "validation_pairs": len(validation_df),
    "test_pairs": len(test_df),
    "search.lr": LEARNING_RATE_RANGE,
    "search.epochs": EPOCHS_RANGE,
    "search.batch_size": BATCH_SIZE_CHOICES,
    "search.weight_decay": WEIGHT_DECAY_RANGE,
    "warmup_ratio": WARMUP_RATIO,
    "pruner": type(PRUNER).__name__,
    "threshold_grid": f"{CLASSIFICATION_THRESHOLDS[0]} - {CLASSIFICATION_THRESHOLDS[-1]}, {len(CLASSIFICATION_THRESHOLDS)} values",
}

The study also remembers the data files it was started with - their hashes are in the context

In [ ]:
data_hash_keys = [f"data_sha256.{path.name}" for path in DATA_PATHS]
study_settings = {
    **run_settings,
    **{key: run_context.environment[key] for key in data_hash_keys},
}

A study or a result folder of other settings stops the run here, before anything is trained or changed.

In [ ]:
runs.check_study_settings(STUDY_PATH, study_settings)
# a result folder with the name of this run has to hold these settings too, and its study -
# without the study the whole search would be trained again
if runs.find_finished_run(RUN_DIR, run_settings) and not STUDY_PATH.is_file():
    raise RuntimeError(
        f"{RUN_DIR} exists but its study {STUDY_PATH.name} does not - move the folder away."
    )

Open the study (or continue it) and store its settings

In [ ]:
study = tuning.open_study(STUDY_PATH, pruner=PRUNER, seed=SEED)
runs.store_study_settings(study, study_settings)

Run the study until it holds N_TRIALS finished trials

In [ ]:
tuning.run_study(
    study,
    partial(
        objective,
        tokenizer=tokenizer,
        tuning_df=tuning_df,
        validation_df=validation_df,
        device=device,
        progress_file=progress_file,
    ),
    n_trials=N_TRIALS,
)

A study with another number of finished trials stops the run

In [ ]:
finished_trials = study.get_trials(deepcopy=False, states=tuning.FINISHED_STATES)
if len(finished_trials) != N_TRIALS:
    raise RuntimeError(
        f"The study holds {len(finished_trials)} finished trials, not {N_TRIALS}."
    )

The best trial - the highest objective value, the earliest trial if two are equal - and the
trial counts

In [ ]:
best_trial = runs.select_best_trial(study)
complete_trials = study.get_trials(deepcopy=False, states=(TrialState.COMPLETE,))
failed_trials = study.get_trials(deepcopy=False, states=(TrialState.FAIL,))
# a crashed attempt (killed pod, OOM) is repeated from its checkpoint - not a real failure
crashed_trials = [trial for trial in failed_trials if trial.user_attrs.get("crashed")]
tuning_training_seconds = runs.sum_training_seconds(
    study, transformer_training.TRAINING_SECONDS_ATTR
)

The trials table

In [ ]:
trials_df = study.trials_dataframe()
trials_df["is_best_trial"] = trials_df["number"] == best_trial.number
# the weighted F1 of the argmax labels on validation after every epoch (step 0 = first epoch), not the trial value
epoch_reports_df = study.trials_dataframe(attrs=("number", "intermediate_values"))
print(
    f"Best trial {best_trial.number}: validation F1 {best_trial.user_attrs['validation_f1']}, {best_trial.params}"
)

# Train the final model

The hyperparameters of the best trial, the training pairs only - the validation pairs are
held out for the threshold of the final model. Every training pair is used once.

In [ ]:
final_settings = transformer_training.TrainingSettings(
    learning_rate=best_trial.params["lr"],
    weight_decay=best_trial.params["weight_decay"],
    epochs=best_trial.params["epochs"],
    batch_size=best_trial.params["batch_size"],
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
)
final_train_df = train_df

The params of the run - the settings and the final hyperparameters

In [ ]:
run_params = {
    **run_settings,
    "final_train_pairs": len(final_train_df),
    "best_trial_number": best_trial.number,
    "best_trial_position": best_trial.user_attrs["position"],
    "final_optimizer_learning_rate": final_settings.learning_rate,
    "final_epochs": final_settings.epochs,
    "final_batch_size": final_settings.batch_size,
    "final_weight_decay": final_settings.weight_decay,
    "final_warmup_ratio": final_settings.warmup_ratio,
}

A result folder marks a finished run - it has to belong to these params

In [ ]:
run_is_complete = runs.find_finished_run(RUN_DIR, run_params)
if run_is_complete:
    print(f"{RUN_DIR} exists - the final model is already trained and evaluated.")

Free the memory of the last trial and load the model

In [ ]:
if not run_is_complete:
    gc.collect()
    torch.cuda.empty_cache()
    model = build_model(device)

Train the final model

In [ ]:
if not run_is_complete:
    progress_file.start_stage("final training")
    final_result = transformer_training.train_classifier(
        model=model,
        tokenizer=tokenizer,
        train_df=final_train_df,
        settings=final_settings,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_key=FINAL_CHECKPOINT_KEY,
        on_step=progress_file.update,
        checkpoint_minutes=CHECKPOINT_MINUTES,
    )

Save the model

In [ ]:
if not run_is_complete:
    model.save_pretrained(MODEL_DIR)

# Evaluate on the test pairs

The final model scores the held-out validation pairs, the threshold is selected on them and
applied to the test pairs unchanged. The test pairs are scored once and never select anything.

Score the validation pairs

In [ ]:
if not run_is_complete:
    progress_file.start_stage("validation scoring", n_pairs=len(validation_df))
    started = time.perf_counter()
    final_validation_scores_df = transformer_training.score_pairs(
        model, tokenizer, validation_df
    )
    scoring_seconds_validation = time.perf_counter() - started

Select the threshold on the validation scores of the final model

In [ ]:
if not run_is_complete:
    final_threshold_results_df = evaluation.evaluate_classification_thresholds(
        final_validation_scores_df["same"].astype(int),
        final_validation_scores_df["score"],
        CLASSIFICATION_THRESHOLDS,
    )
    threshold = float(
        evaluation.select_best_threshold(final_threshold_results_df)["threshold"]
    )

Score the test pairs once, with that threshold

In [ ]:
if not run_is_complete:
    progress_file.start_stage("test scoring", n_pairs=len(test_df))
    started = time.perf_counter()
    test_scores_df = transformer_training.score_pairs(model, tokenizer, test_df)
    scoring_seconds_test = time.perf_counter() - started
    predictions_df = runs.build_predictions(test_df, test_scores_df, threshold)

The tables of the run

In [ ]:
if not run_is_complete:
    run_tables = {
        "trials": trials_df,
        "predictions-validation": runs.build_predictions(
            validation_df, final_validation_scores_df, threshold
        ),
        "predictions-test": predictions_df,
        "trial-epoch-reports": epoch_reports_df,
        # the validation F1 of the final model at every threshold of the grid
        "thresholds-validation": final_threshold_results_df,
    }

The test metrics

In [ ]:
if not run_is_complete:
    test_metrics = evaluation.evaluate_results(
        predictions_df["same"], predictions_df["pred"]
    )
    print(f"Threshold {threshold}: {test_metrics}")

Free the memory of the final model

In [ ]:
if not run_is_complete:
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Create the confusion matrix

The figure

In [ ]:
if not run_is_complete:
    confusion_figure, axis = plt.subplots(figsize=(6, 5))

The counts of the test pairs

In [ ]:
if not run_is_complete:
    sns.heatmap(
        confusion_matrix(predictions_df["same"], predictions_df["pred"], labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[False, True],
        yticklabels=[False, True],
        ax=axis,
        cbar=False,
    )

The labels

In [ ]:
if not run_is_complete:
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_title(f"Confusion Matrix: {MODEL_ID}")

# Save the results

Everything the run reports is written into one folder; it is uploaded to MLflow from there.

The metrics of the run

In [ ]:
if not run_is_complete:
    run_metrics = {
        "threshold": threshold,
        # logged only - the best trial's threshold is never applied
        "best_trial_threshold": best_trial.user_attrs["threshold"],
        **{f"test_{name}": value for name, value in test_metrics.to_dict().items()},
        "best_trial_validation_f1": best_trial.user_attrs["validation_f1"],
        "n_trials_complete": len(complete_trials),
        "n_trials_pruned": len(finished_trials) - len(complete_trials),
        "n_trials_failed": len(failed_trials) - len(crashed_trials),
        "n_trials_crashed": len(crashed_trials),
        "training_seconds_tuning": tuning_training_seconds,
        "training_seconds_final": final_result.training_seconds,
        "scoring_seconds_validation": round(scoring_seconds_validation, 1),
        "scoring_seconds_test": round(scoring_seconds_test, 1),
        # the peak since the last pod start - its trials, the final training and the scoring
        "max_memory_allocated_gb": round(
            torch.cuda.max_memory_allocated(device) / 1024**3, 2
        )
        if device.type == "cuda"
        else 0.0,
        **{
            f"final_loss_epoch_{epoch}": loss
            for epoch, loss in enumerate(final_result.epoch_losses, start=1)
        },
    }

Write the result folder

In [ ]:
if not run_is_complete:
    runs.remove_unfinished_run_dir(RUN_DIR)
    tracking.write_run_files(
        RUN_DIR,
        context=run_context,
        params=run_params,
        metrics=run_metrics,
        tables=run_tables,
        figures={"confusion_matrix-test": confusion_figure},
    )

The finished run needs no checkpoint any more

In [ ]:
(CHECKPOINT_DIR / transformer_training.CHECKPOINT_FILE).unlink(missing_ok=True)
progress_file.start_stage("done")
print(RUN_DIR)
print((RUN_DIR / tracking.METRICS_FILE).read_text(encoding="utf-8"))